In [1]:
from qick import QickSoc

soc = QickSoc(bitfile="/home/xilinx/qick_repo/qick_lib/qick/qick_4x2.bit")
print(soc)

QICK running on RFSoC4x2, software version 0.2.388

Firmware configuration (built Wed Sep  6 18:49:29 2023):

	Global clocks (MHz): tProc dispatcher timing 409.600, RF reference 491.520
	Groups of related clocks: [tProc clock, DAC tile 0], [DAC tile 2], [ADC tile 0]

	2 signal generator channels:
	0:	axis_signal_gen_v6 - fs=9830.400 Msps, fabric=614.400 MHz
		envelope memory: 65536 complex samples (6.667 us)
		32-bit DDS, range=9830.400 MHz
		DAC tile 0, blk 0 is DAC_B
	1:	axis_signal_gen_v6 - fs=9830.400 Msps, fabric=614.400 MHz
		envelope memory: 65536 complex samples (6.667 us)
		32-bit DDS, range=9830.400 MHz
		DAC tile 2, blk 0 is DAC_A

	2 readout channels:
	0:	axis_readout_v2 - configured by PYNQ
		fs=4423.680 Msps, decimated=552.960 MHz, 32-bit DDS, range=4423.680 MHz
		axis_avg_buffer v1.0 (no edge counter, no weights)
		memory 16384 accumulated, 1024 decimated (1.852 us)
		triggered by output 7, pin 14, feedback to tProc input 0
		ADC tile 0, blk 0 is ADC_D
	1:	axis_readout_v

In [3]:
from qick import QickSoc
from qick.averager_program import AveragerProgram
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import datetime
import os

os.makedirs("/home/xilinx/jupyter_notebooks/spectrum-analyzer", exist_ok=True)

class RawCaptureProgram(AveragerProgram):
    def initialize(self):
        cfg = self.cfg
        self.declare_readout(
            ch=cfg["ro_ch"],
            length=cfg["readout_length"],
            freq=0,
            gen_ch=None
        )
        self.synci(200)

    def body(self):
        self.trigger(
            adcs=[self.cfg["ro_ch"]],
            pins=[0],
            adc_trig_offset=self.cfg["adc_trig_offset"]
        )
        self.wait_all()
        self.sync_all(self.us2cycles(self.cfg["relax_delay"]))

cfg = {
    "ro_ch":           0,
    "readout_length":  1000,
    "adc_trig_offset": 100,
    "soft_avgs":       1,
    "reps":            1,
    "relax_delay":     1.0,
}

prog    = RawCaptureProgram(soc, cfg)
iq_list = prog.acquire_decimated(soc, load_pulses=False, progress=False)

i_data = iq_list[0][0]
q_data = iq_list[0][1]

print(f"[RAW ADC] Captured {len(i_data)} I samples, {len(q_data)} Q samples")
print(f"[RAW ADC] Data type : {i_data.dtype}")
print(f"[RAW ADC] I range   : {i_data.min():.1f} to {i_data.max():.1f}")
print(f"[RAW ADC] Q range   : {q_data.min():.1f} to {q_data.max():.1f}")
print(f"[RAW ADC] PASS — direct ADC samples confirmed, no DDC")

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import datetime

# Correct way to get ADC sample rate on PYNQ 3.0.1
fs_mhz = soc['readouts'][0]['fs']

N         = len(i_data)
samples   = i_data + 1j * q_data
spec_db   = 10 * np.log10(
    np.abs(np.fft.fftshift(np.fft.fft(samples)))**2 + 1e-20)
freqs_mhz = np.fft.fftshift(np.fft.fftfreq(N, d=1.0/fs_mhz))

fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor('#0d0d0d')
ax.set_facecolor('#0d0d0d')
ax.plot(freqs_mhz, spec_db, color='#00e5ff', linewidth=0.5)
ax.set_xlabel('Frequency (MHz)', color='white')
ax.set_ylabel('Power (dB)',      color='white')
ax.set_title(
    f'QICK Raw ADC — PYNQ 3.0.1 — fs={fs_mhz:.1f} MHz | '
    f'Nyquist={fs_mhz/2:.1f} MHz | N={N}',
    color='white')
ax.tick_params(colors='white')
ax.spines[:].set_color('#333333')
ax.grid(True, color='#1e1e1e', linewidth=0.4)
plt.tight_layout()

ts  = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
out = f"/home/xilinx/jupyter_notebooks/spectrum-analyzer/qick_raw_adc_{ts}.png"
fig.savefig(out, dpi=100, bbox_inches='tight', facecolor='#0d0d0d')
plt.close(fig)
print(f"[PLOT] Saved: {out}")
print(f"[INFO] ADC sample rate: {fs_mhz:.3f} MHz")
print(f"[INFO] Nyquist: {fs_mhz/2:.3f} MHz")

[RAW ADC] Captured 1000 I samples, 1000 Q samples
[RAW ADC] Data type : float64
[RAW ADC] I range   : -6.0 to 8.0
[RAW ADC] Q range   : 0.0 to 0.0
[RAW ADC] PASS — direct ADC samples confirmed, no DDC
[PLOT] Saved: /home/xilinx/jupyter_notebooks/spectrum-analyzer/qick_raw_adc_20250530_101604.png
[INFO] ADC sample rate: 4423.680 MHz
[INFO] Nyquist: 2211.840 MHz


In [4]:
# Keep only positive frequencies — 0 to Nyquist
pos_mask  = freqs_mhz >= 0
freq_plot = freqs_mhz[pos_mask]
spec_plot = spec_db[pos_mask]

fig, ax = plt.subplots(figsize=(14, 4))
fig.patch.set_facecolor('#0d0d0d')
ax.set_facecolor('#0d0d0d')
ax.plot(freq_plot, spec_plot, color='#00e5ff', linewidth=0.5)
ax.set_xlim(0, fs_mhz / 2)
ax.set_xlabel('Frequency (MHz)', color='white')
ax.set_ylabel('Power (dB)',      color='white')
ax.set_title(
    f'QICK Raw ADC — PYNQ 3.0.1 — 0 to {fs_mhz/2:.1f} MHz | N={N}',
    color='white')
ax.tick_params(colors='white')
ax.spines[:].set_color('#333333')
ax.grid(True, color='#1e1e1e', linewidth=0.4)
plt.tight_layout()

ts  = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
out = f"/home/xilinx/jupyter_notebooks/spectrum-analyzer/qick_raw_adc_{ts}.png"
fig.savefig(out, dpi=100, bbox_inches='tight', facecolor='#0d0d0d')
plt.close(fig)
print(f"[PLOT] Saved: {out}")

[PLOT] Saved: /home/xilinx/jupyter_notebooks/spectrum-analyzer/qick_raw_adc_20250530_101833.png


In [9]:
from qick.averager_program import AveragerProgram
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import datetime

class LoopbackToneProgram(AveragerProgram):
    def initialize(self):
        cfg = self.cfg
        self.declare_gen(ch=cfg["gen_ch"], nqz=1)
        self.declare_readout(
            ch=cfg["ro_ch"],
            length=cfg["readout_length"],
            freq=cfg["f_tone"],
            gen_ch=cfg["gen_ch"]
        )
        self.set_pulse_registers(
            ch=cfg["gen_ch"],
            style="const",
            freq=cfg["f_tone"],
            phase=0,
            gain=cfg["gain"],
            length=cfg["readout_length"] + 200
        )
        self.synci(200)

    def body(self):
        self.trigger(
            adcs=[self.cfg["ro_ch"]],
            pins=[0],
            adc_trig_offset=self.cfg["adc_trig_offset"]
        )
        self.pulse(ch=self.cfg["gen_ch"])
        self.wait_all()
        self.sync_all(self.us2cycles(self.cfg["relax_delay"]))

# Use the DECIMATED sample rate for the frequency axis
fs_decimated_mhz = 552.960   # from print(soc) output

# Only sweep frequencies within the decimated bandwidth
# Decimated Nyquist = 552.960 / 2 = 276.48 MHz
SWEEP_FREQS_MHZ = [10, 20, 50, 100, 150, 200, 250]

cfg = {
    "gen_ch":          0,
    "ro_ch":           0,
    "adc_trig_offset": 100,
    "soft_avgs":       20,
    "reps":            1,
    "relax_delay":     1.0,
    "readout_length":  1000,
    "gain":            5000,
    "f_tone":          100,
}

print(f"[CW SWEEP] DAC_B → ADC_C loopback")
print(f"[CW SWEEP] Decimated fs = {fs_decimated_mhz:.3f} MHz")
print(f"[CW SWEEP] Decimated Nyquist = {fs_decimated_mhz/2:.3f} MHz\n")
print(f"{'f_inject (MHz)':>16} | {'f_measured (MHz)':>18} | "
      f"{'Error (MHz)':>12} | {'SNR (dB)':>10}")
print("-" * 65)

results = []

for f_tone in SWEEP_FREQS_MHZ:
    cfg["f_tone"] = f_tone
    prog = LoopbackToneProgram(soc, cfg)
    iq   = prog.acquire_decimated(soc, load_pulses=True, progress=False)

    i_data  = iq[0][0]
    q_data  = iq[0][1]
    samples = i_data + 1j * q_data
    N       = len(samples)

    spectrum = np.abs(np.fft.fftshift(np.fft.fft(samples)))**2
    freqs    = np.fft.fftshift(
                   np.fft.fftfreq(N, d=1.0/fs_decimated_mhz))

    peak_idx   = np.argmax(spectrum)
    f_measured = abs(float(freqs[peak_idx]))
    noise      = float(np.median(spectrum))
    snr        = 10 * np.log10(
                     float(spectrum[peak_idx]) / (noise + 1e-20))
    error      = f_measured - f_tone

    flag = "✓" if abs(error) < 2.0 else "✗"
    print(f"{f_tone:>16.1f} | {f_measured:>18.2f} | "
          f"{error:>+12.2f} | {snr:>10.1f}  {flag}")
    results.append((f_tone, f_measured, error, snr))

n_pass = sum(1 for _, _, e, _ in results if abs(e) < 2.0)
print(f"\n[RESULT] {n_pass}/{len(results)} tones within ±2 MHz")
if n_pass == len(results):
    print("[PASS] CW calibration working correctly")
else:
    print("[PARTIAL] Checking frequency axis...")
    # Print diagnostic info to help identify the scaling factor
    print(f"\n[DIAG] Ratio of measured/injected for first tone:")
    print(f"       {results[0][1]:.2f} / {results[0][0]:.2f} = "
          f"{results[0][1]/results[0][0]:.4f}")
    print(f"       This ratio should be ~1.0 if axis is correct")

[CW SWEEP] DAC_B → ADC_C loopback
[CW SWEEP] Decimated fs = 552.960 MHz
[CW SWEEP] Decimated Nyquist = 276.480 MHz

  f_inject (MHz) |   f_measured (MHz) |  Error (MHz) |   SNR (dB)
-----------------------------------------------------------------
            10.0 |               9.95 |        -0.05 |       25.4  ✓
            20.0 |              19.91 |        -0.09 |       22.5  ✓
            50.0 |              49.77 |        -0.23 |       19.3  ✓
           100.0 |              95.66 |        -4.34 |       16.2  ✗
           150.0 |             149.85 |        -0.15 |       23.5  ✓
           200.0 |             194.09 |        -5.91 |       15.1  ✗
           250.0 |             160.91 |       -89.09 |       12.3  ✗

[RESULT] 4/7 tones within ±2 MHz
[PARTIAL] Checking frequency axis...

[DIAG] Ratio of measured/injected for first tone:
       9.95 / 10.00 = 0.9953
       This ratio should be ~1.0 if axis is correct


In [10]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import datetime
import numpy as np

# Final validation sweep — stay within safe decimated bandwidth
SAFE_FREQS_MHZ = [10, 20, 50, 75, 100, 125, 150]

cfg_val = {
    "gen_ch":          0,
    "ro_ch":           0,
    "adc_trig_offset": 100,
    "soft_avgs":       20,
    "reps":            1,
    "relax_delay":     1.0,
    "readout_length":  1000,
    "gain":            5000,
    "f_tone":          100,
}

fs_decimated_mhz = 552.960
results_val = []

print(f"[VALIDATION] CW loopback sweep — safe bandwidth 0–150 MHz")
print(f"{'f_inject':>12} | {'f_measured':>12} | {'error':>10} | "
      f"{'SNR':>8}")
print("-" * 50)

for f_tone in SAFE_FREQS_MHZ:
    cfg_val["f_tone"] = f_tone
    prog = LoopbackToneProgram(soc, cfg_val)
    iq   = prog.acquire_decimated(soc, load_pulses=True, progress=False)

    samples  = iq[0][0] + 1j * iq[0][1]
    N        = len(samples)
    spectrum = np.abs(np.fft.fftshift(np.fft.fft(samples)))**2
    freqs    = np.fft.fftshift(np.fft.fftfreq(N, d=1.0/fs_decimated_mhz))

    pos_mask   = freqs >= 0
    spec_p     = spectrum[pos_mask]
    freqs_p    = freqs[pos_mask]
    peak_idx   = np.argmax(spec_p)
    f_measured = float(freqs_p[peak_idx])
    noise      = float(np.median(spec_p))
    snr        = 10 * np.log10(float(spec_p[peak_idx]) / (noise + 1e-20))
    error      = f_measured - f_tone

    flag = "✓" if abs(error) < 2.0 else "✗"
    print(f"{f_tone:>11.1f}  | {f_measured:>11.2f}  | "
          f"{error:>+9.2f}  | {snr:>7.1f}  {flag}")
    results_val.append((f_tone, f_measured, error, snr))

n_pass = sum(1 for _, _, e, _ in results_val if abs(e) < 2.0)
print(f"\n[RESULT] {n_pass}/{len(results_val)} tones within ±2 MHz")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d0d0d')

ax1 = axes[0]
ax1.set_facecolor('#0d0d0d')
f_inj  = [r[0] for r in results_val]
f_meas = [r[1] for r in results_val]
snrs   = [r[3] for r in results_val]
ax1.plot([0, 160], [0, 160], color='#444444',
         linewidth=1, linestyle='--', label='Ideal')
sc = ax1.scatter(f_inj, f_meas, c=snrs, cmap='RdYlGn',
                 s=100, vmin=0, vmax=30, zorder=3)
plt.colorbar(sc, ax=ax1, label='SNR (dB)').ax.tick_params(colors='white')
ax1.set_xlabel('Injected frequency (MHz)', color='white')
ax1.set_ylabel('Measured frequency (MHz)', color='white')
ax1.set_title('CW Calibration — Measured vs Injected',
              color='white')
ax1.tick_params(colors='white')
ax1.spines[:].set_color('#333333')
ax1.grid(True, color='#1e1e1e', linewidth=0.4)
ax1.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8)

ax2 = axes[1]
ax2.set_facecolor('#0d0d0d')
errors = [r[2] for r in results_val]
ax2.bar(f_inj, errors, color=['#00e5ff' if abs(e) < 2.0
        else '#ff6b35' for e in errors], width=8)
ax2.axhline(0,  color='white',   linewidth=0.8, linestyle='--')
ax2.axhline(2,  color='#88ff88', linewidth=0.8,
            linestyle=':', label='±2 MHz limit')
ax2.axhline(-2, color='#88ff88', linewidth=0.8, linestyle=':')
ax2.set_xlabel('Injected frequency (MHz)', color='white')
ax2.set_ylabel('Frequency error (MHz)',    color='white')
ax2.set_title(f'CW Error | {n_pass}/{len(results_val)} pass',
              color='white')
ax2.tick_params(colors='white')
ax2.spines[:].set_color('#333333')
ax2.grid(True, color='#1e1e1e', linewidth=0.4)
ax2.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8)

plt.tight_layout()
ts  = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
out = (f"/home/xilinx/jupyter_notebooks/spectrum-analyzer/"
       f"qick_cw_validation_{ts}.png")
fig.savefig(out, dpi=100, bbox_inches='tight', facecolor='#0d0d0d')
plt.close(fig)
print(f"[PLOT] Saved: {out}")

if n_pass == len(results_val):
    print("\n✅ QICK FULLY VALIDATED")
    print("   — Raw ADC samples confirmed")
    print("   — CW loopback calibration passing")
    print("   — Ready for wideband survey and PFB/FFT investigation")

[VALIDATION] CW loopback sweep — safe bandwidth 0–150 MHz
    f_inject |   f_measured |      error |      SNR
--------------------------------------------------
       10.0  |        9.95  |     -0.05  |    24.7  ✓
       20.0  |       19.91  |     -0.09  |    24.2  ✓
       50.0  |       49.77  |     -0.23  |    17.4  ✓
       75.0  |       75.20  |     +0.20  |    23.4  ✓
      100.0  |      100.09  |     +0.09  |    25.1  ✓
      125.0  |      127.18  |     +2.18  |    14.2  ✗
      150.0  |      149.85  |     -0.15  |    25.5  ✓

[RESULT] 6/7 tones within ±2 MHz
[PLOT] Saved: /home/xilinx/jupyter_notebooks/spectrum-analyzer/qick_cw_validation_20250530_102750.png


In [13]:
from qick.averager_program import AveragerProgram
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import time
import datetime
import os

os.makedirs("/home/xilinx/jupyter_notebooks/spectrum-analyzer", exist_ok=True)

class SpectrumProgram(AveragerProgram):
    def initialize(self):
        cfg = self.cfg
        self.declare_readout(
            ch=cfg["ro_ch"],
            length=cfg["readout_length"],
            freq=0,
            gen_ch=None
        )
        self.synci(200)

    def body(self):
        self.trigger(
            adcs=[self.cfg["ro_ch"]],
            pins=[0],
            adc_trig_offset=self.cfg["adc_trig_offset"]
        )
        self.wait_all()
        self.sync_all(self.us2cycles(self.cfg["relax_delay"]))

# ── Configuration ──────────────────────────────────────────────
OBSERVATION_SECONDS = 300
FRAMES_PER_SECOND   = 3
MAX_WATERFALL_ROWS  = 300
SAVE_PATH = "/home/xilinx/jupyter_notebooks/spectrum-analyzer/"

cfg = {
    "ro_ch":           0,
    "readout_length":  1000,
    "adc_trig_offset": 100,
    "soft_avgs":       50,
    "reps":            1,
    "relax_delay":     1.0,
}

fs_mhz  = 4423.680
N       = cfg["readout_length"]
nyquist = fs_mhz / 2

# Frequency axis — positive side only
freqs_full = np.fft.fftshift(np.fft.fftfreq(N, d=1.0/fs_mhz))
pos_mask   = freqs_full >= 0
freq_plot  = freqs_full[pos_mask]
n_bins     = len(freq_plot)

print(f"[INFO] ADC sample rate  : {fs_mhz:.1f} MHz")
print(f"[INFO] Nyquist          : {nyquist:.1f} MHz")
print(f"[INFO] Freq resolution  : {fs_mhz/N*1000:.1f} kHz/bin")
print(f"[INFO] Observation time : {OBSERVATION_SECONDS}s at "
      f"{FRAMES_PER_SECOND} fps")
print(f"[INFO] Remove antenna loopback cable — connect antenna to ADC_C\n")

# Known RFI bands
RFI = {
    'FM\n(88-108)':      (88,   108,  '#aaffaa'),
    'DAB\n(174-240)':    (174,  240,  '#aaffaa'),
    '4G\n(700-960)':     (700,  960,  '#ffaaff'),
    'GPS\n(1575)':       (1570, 1580, '#aaaaff'),
    '3G/4G\n(2100)':     (2100, 2170, '#ffaaff'),
}
WH_FM  = [88.4, 90.4, 94.6, 96.0, 97.6, 99.4, 102.0, 103.0]
WH_DAB = [209.936, 222.064]

# ── Acquisition loop ───────────────────────────────────────────
ts        = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
prog      = SpectrumProgram(soc, cfg)
max_hold  = np.zeros(n_bins, dtype=np.float32)
stored    = []
n_frames  = 0
n_clipped = 0
interval  = 1.0 / FRAMES_PER_SECOND
t_start   = time.time()
t_next    = t_start

print(f"[ACQ] Starting {OBSERVATION_SECONDS}s observation UTC {ts}")

while (time.time() - t_start) < OBSERVATION_SECONDS:
    now = time.time()
    if now >= t_next:
        iq       = prog.acquire_decimated(soc, load_pulses=False,
                                          progress=False)
        samples  = iq[0][0] + 1j * iq[0][1]
        spectrum = np.abs(np.fft.fftshift(np.fft.fft(samples)))**2
        spec_db  = 10 * np.log10(
                       spectrum[pos_mask] + 1e-20).astype(np.float32)

        max_hold  = np.maximum(max_hold, spec_db)
        n_frames += 1
        t_next    = t_start + n_frames * interval

        if np.max(np.abs(iq[0][0])) > 0.95 * 32767:
            n_clipped += 1

        if len(stored) < MAX_WATERFALL_ROWS:
            stored.append(spec_db.copy())

        if n_frames % 30 == 0:
            elapsed  = now - t_start
            clip_pct = n_clipped / n_frames * 100
            print(f"  {n_frames:4d} frames | {elapsed:5.0f}s elapsed | "
                  f"{OBSERVATION_SECONDS-elapsed:5.0f}s remaining | "
                  f"clip: {clip_pct:.1f}%")

    time.sleep(0.05)

elapsed_total = time.time() - t_start
clip_fraction = n_clipped / n_frames * 100 if n_frames > 0 else 0

print(f"\n[DONE] {n_frames} frames in {elapsed_total:.1f}s")
print(f"[CLIP] Clipping fraction: {clip_fraction:.2f}%")

# Save arrays
data_wf = np.array(stored, dtype=np.float32)
np.save(f"{SAVE_PATH}qick_maxhold_{ts}.npy",   max_hold)
np.save(f"{SAVE_PATH}qick_waterfall_{ts}.npy", data_wf)
np.save(f"{SAVE_PATH}qick_freqaxis_{ts}.npy",
        freq_plot.astype(np.float32))
print(f"[SAVE] Arrays saved (waterfall shape: {data_wf.shape})")

# ── Plot ───────────────────────────────────────────────────────
mean_spectrum = np.mean(data_wf, axis=0)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.patch.set_facecolor('#0d0d0d')

ax1 = axes[0]
ax1.set_facecolor('#0d0d0d')
ax1.plot(freq_plot, mean_spectrum,
         color='#00e5ff', linewidth=0.5, alpha=0.8,
         label='Mean', zorder=2)
ax1.plot(freq_plot, max_hold,
         color='#ff6b35', linewidth=0.6,
         label='Max hold', zorder=3)

for label, (f_lo, f_hi, colour) in RFI.items():
    if f_lo < freq_plot[-1]:
        ax1.axvspan(f_lo, min(f_hi, freq_plot[-1]),
                    alpha=0.08, color=colour, zorder=1)
        ax1.text((f_lo + min(f_hi, freq_plot[-1])) / 2,
                 max_hold.max() + 1, label,
                 color=colour, fontsize=6,
                 ha='center', va='bottom')

for f in WH_FM:
    ax1.axvline(f, color='#88ff88', linewidth=0.8,
                linestyle='--', alpha=0.6, zorder=4)
ax1.axvline(WH_FM[0], color='#88ff88', linewidth=0.8,
            linestyle='--', alpha=0.6,
            label='Winterhill FM', zorder=4)
for f in WH_DAB:
    ax1.axvline(f, color='#ffff88', linewidth=0.8,
                linestyle=':', alpha=0.6, zorder=4)
ax1.axvline(WH_DAB[0], color='#ffff88', linewidth=0.8,
            linestyle=':', alpha=0.6,
            label='Winterhill DAB', zorder=4)

ax1.set_xlim(0, nyquist)
ax1.set_ylabel('Power (dB)', color='white', fontsize=11)
ax1.set_title(
    f'Wide-Band Spectrum — QICK Direct Sampling | RFSoC 4x2 | '
    f'UTC {ts} | {elapsed_total:.0f}s | {n_frames} frames | '
    f'Clip: {clip_fraction:.2f}%',
    color='white', fontsize=10)
ax1.tick_params(colors='white')
ax1.spines[:].set_color('#333333')
ax1.grid(True, color='#1e1e1e', linewidth=0.4)
ax1.xaxis.set_major_locator(ticker.MultipleLocator(200))
ax1.legend(facecolor='#1a1a1a', edgecolor='#444',
           labelcolor='white', fontsize=8, loc='upper right')

ax2 = axes[1]
ax2.set_facecolor('#0d0d0d')
wf = ax2.imshow(
    data_wf,
    aspect='auto',
    extent=[0, nyquist, elapsed_total, 0],
    cmap='inferno',
    vmin=np.percentile(data_wf, 2),
    vmax=np.percentile(data_wf, 98),
)
cb = plt.colorbar(wf, ax=ax2, label='Power (dB)', pad=0.01)
cb.ax.yaxis.label.set_color('white')
cb.ax.tick_params(colors='white')
for label, (f_lo, f_hi, colour) in RFI.items():
    if f_lo < freq_plot[-1]:
        ax2.axvspan(f_lo, min(f_hi, freq_plot[-1]),
                    alpha=0.06, color=colour)
ax2.set_xlabel('Frequency (MHz)', color='white', fontsize=11)
ax2.set_ylabel('Time (s)',        color='white', fontsize=11)
ax2.set_title(
    f'Waterfall ({len(data_wf)} rows | '
    f'QICK direct sampling 0–{nyquist:.0f} MHz)',
    color='white', fontsize=10)
ax2.tick_params(colors='white')
ax2.xaxis.set_major_locator(ticker.MultipleLocator(200))

plt.tight_layout()
out = f"{SAVE_PATH}qick_survey_{ts}.png"
fig.savefig(out, dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
plt.close(fig)
print(f"[PLOT] Saved: {out}")
print(f"\nDownload with:")
print(f"  scp xilinx@192.168.3.1:{SAVE_PATH}qick_survey_{ts}.png .")


[INFO] ADC sample rate  : 4423.7 MHz
[INFO] Nyquist          : 2211.8 MHz
[INFO] Freq resolution  : 4423.7 kHz/bin
[INFO] Observation time : 300s at 3 fps
[INFO] Remove antenna loopback cable — connect antenna to ADC_C

[ACQ] Starting 300s observation UTC 20250530_104203
    30 frames |    10s elapsed |   290s remaining | clip: 0.0%
    60 frames |    20s elapsed |   280s remaining | clip: 0.0%
    90 frames |    30s elapsed |   270s remaining | clip: 0.0%
   120 frames |    40s elapsed |   260s remaining | clip: 0.0%
   150 frames |    50s elapsed |   250s remaining | clip: 0.0%
   180 frames |    60s elapsed |   240s remaining | clip: 0.0%
   210 frames |    70s elapsed |   230s remaining | clip: 0.0%
   240 frames |    80s elapsed |   220s remaining | clip: 0.0%
   270 frames |    90s elapsed |   210s remaining | clip: 0.0%
   300 frames |   100s elapsed |   200s remaining | clip: 0.0%
   330 frames |   110s elapsed |   190s remaining | clip: 0.0%
   360 frames |   120s elapsed |   